# Kuis 1 — pandas dasar (Pertemuan 2)
**Waktu: 20 menit · Individu · Bobot: 3% dari nilai akhir**

**Aturan:**
- Boleh membuka catatan, notebook praktikum, dan dokumentasi pandas.
- **Tidak boleh** menggunakan asisten AI, dan tidak boleh berdiskusi dengan mahasiswa lain.
- Ganti nama file menjadi `P02_Kuis1_<NIM>_<Nama>.ipynb`.
- Kumpulkan: *Restart & Run all* → pastikan tanpa error → unduh `.ipynb` → unggah ke LMS.

**Data:** `kunjungan_perpustakaan_2025.csv` — catatan kunjungan perpustakaan kampus, Februari–Juni 2025.
Satu baris = satu kunjungan.

In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/Exploratory Data Analistic/Data/kunjungan_perpustakaan_2025.csv')
df.head()

,id_kunjungan,tanggal,nim_hash,prodi,angkatan,durasi_menit,jumlah_pinjam,jenis_ruang,kepuasan
0,K100000,2025-05-18,H1160,Sistem Informasi,2024,30,0,Lab Komputer,5.0
1,K100001,2025-02-22,H1190,Teknik Informatika,2023,78,1,Ruang Baca,3.6
2,K100002,2025-06-11,H1070,Sistem Informasi,2025,56,2,Ruang Baca,4.1
3,K100003,2025-05-06,H1001,Sistem Informasi,2024,12,1,Lab Komputer,2.6
4,K100004,2025-02-24,H1047,Teknik Informatika,2023,67,1,Lab Komputer,4.3


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---
### Tugas 1 (15 poin) — Memuat dan memeriksa

In [8]:
# a. Cetak jumlah baris dan kolom
print("--- a. Jumlah Baris dan Kolom ---")
print(f"Baris: {df.shape[0]}, Kolom: {df.shape[1]}")

# b. Cetak jumlah nilai kosong (missing) untuk setiap kolom
print("\n--- b. Jumlah Nilai Kosong per Kolom ---")
print(df.isnull().sum())

# c. Cetak berapa mahasiswa unik (nim_hash) yang pernah berkunjung
print("\n--- c. Jumlah Mahasiswa Unik ---")
print(f"Jumlah mahasiswa unik: {df['nim_hash'].nunique()}")


--- a. Jumlah Baris dan Kolom ---
Baris: 900, Kolom: 9

--- b. Jumlah Nilai Kosong per Kolom ---
id_kunjungan      0
tanggal           0
nim_hash          0
prodi             0
angkatan          0
durasi_menit      0
jumlah_pinjam     0
jenis_ruang       0
kepuasan         45
dtype: int64

--- c. Jumlah Mahasiswa Unik ---
Jumlah mahasiswa unik: 198


---
### Tugas 2 (20 poin) — Filter

In [9]:
# Filter kunjungan prodi 'Teknik Informatika' dengan durasi_menit > 120
ti_lama = df[(df["prodi"] == "Teknik Informatika") & (df["durasi_menit"] > 120)]

# a. Hitung jumlah baris
jumlah_baris = len(ti_lama)

# b. Hitung rata-rata jumlah_pinjam (2 desimal)
rata_pinjam = ti_lama["jumlah_pinjam"].mean()

print(f"Jumlah baris: {jumlah_baris}")
print(f"Rata-rata jumlah pinjam: {rata_pinjam:.2f}")


Jumlah baris: 50
Rata-rata jumlah pinjam: 1.02


---
### Tugas 3 (20 poin) — Kolom turunan

In [15]:
# a. Buat kolom 'durasi_jam' dibulatkan 2 desimal
df["durasi_jam"] = (df["durasi_menit"] / 60).round(2)

# b. Buat kolom 'lama' ('Panjang' jika durasi_menit >= 90, selain itu 'Singkat')
df["lama"] = df["durasi_menit"].apply(lambda x: "Panjang" if x >= 90 else "Singkat")

# c. Tampilkan 5 kunjungan dengan durasi terlama (kolom: prodi, jenis_ruang, durasi_jam)
top5_durasi = df.nlargest(5, "durasi_menit")[
    ["prodi", "jenis_ruang", "durasi_jam"]
]
print(top5_durasi)

                  prodi    jenis_ruang  durasi_jam
283      Bisnis Digital     Ruang Baca        3.07
533  Teknik Informatika  Ruang Diskusi        2.95
749  Teknik Informatika  Ruang Diskusi        2.82
756  Teknik Informatika     Ruang Baca        2.78
542  Teknik Informatika     Ruang Baca        2.77


---
### Tugas 4 (25 poin) — Agregasi

In [ ]:
# Buat tabel ringkasan per 'jenis_ruang' yang memuat:
#   - jumlah kunjungan
#   - rata-rata durasi_menit (1 desimal)
#   - total jumlah_pinjam
#   - rata-rata kepuasan (2 desimal)
# Urutkan dari rata-rata durasi terbesar. Simpan ke variabel 'ringkas' dan tampilkan.


In [12]:
ringkas = df.groupby('jenis_ruang').agg(
    jumlah_kunjungan=('id_kunjungan', 'count'),
    rata_rata_durasi_menit=('durasi_menit', lambda x: x.mean().round(1)),
    total_jumlah_pinjam=('jumlah_pinjam', 'sum'),
    rata_rata_kepuasan=('kepuasan', lambda x: x.mean().round(2))
).sort_values(by='rata_rata_durasi_menit', ascending=False)

display(ringkas)

,jumlah_kunjungan,rata_rata_durasi_menit,total_jumlah_pinjam,rata_rata_kepuasan
jenis_ruang,,,,
Ruang Baca,405,77.2,452,4.00
Lab Komputer,184,73.4,203,3.96
Ruang Diskusi,311,72.9,338,3.97


---
### Tugas 5 (10 poin) — Menyimpan hasil

In [14]:
# Simpan tabel 'ringkas' ke file 'ringkasan_ruang.csv'
# Buat tabel ringkasan berdasarkan jenis_ruang
ringkas = (
    df.groupby("jenis_ruang")
    .agg(
        total_kunjungan=("id_kunjungan", "count"),
        rata_durasi_menit=("durasi_menit", "mean"),
        total_pinjam=("jumlah_pinjam", "sum"),
    )
    .reset_index()
)

# Simpan ke file 'ringkasan_ruang.csv'
ringkas.to_csv("ringkasan_ruang.csv", index=False)

print("Tabel ringkasan berhasil disimpan ke 'ringkasan_ruang.csv':")
print(ringkas)


Tabel ringkasan berhasil disimpan ke 'ringkasan_ruang.csv':
     jenis_ruang  total_kunjungan  rata_durasi_menit  total_pinjam
0   Lab Komputer              184          73.353261           203
1     Ruang Baca              405          77.150617           452
2  Ruang Diskusi              311          72.916399           338


---
### Tugas 6 (10 poin) — Interpretasi
Tulis **3–4 kalimat** berdasarkan tabel Tugas 4:
1. Apa pola yang Anda lihat?
2. Apa SATU hal yang belum bisa Anda simpulkan dari data ini, dan mengapa?

 Dominasi Mahasiswa Teknik Informatika, Ruang Baca Menjadi Tempat Favorit dan Terbanyak, Konsisten Tingkat Kepuasan,dll.
Tujuan Mahasiswa karna tidak ada variabel atau dekripsi